# Coverage of the Arthouse Working Set
## From Analytics to Action — DTU Spring 2026

**Working set:** films with `arthouse_score >= 7` from `films_arthouse_scored.csv`. Anything 6 or below is out.

**Question:** for the films we *do* care about, how many have the data we'd need for a real analysis — IMDb ratings, plot text, budget/revenue, MovieLens ratings?

## 1. Load and filter

In [ ]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / '03-data').exists() and (candidate / 'src').exists():
        PROJECT_ROOT = candidate
        break
os.chdir(PROJECT_ROOT)

raw = pd.read_csv('notebooks/arthouse/arthouse-LLM-classification/films_arthouse_scored.csv')
df = raw[raw['arthouse_score'] >= 7].copy()

print(f'Total scored films:   {len(raw):,}')
print(f'Arthouse (score ≥ 7): {len(df):,}  ({len(df)/len(raw)*100:.1f}%)')

score_dist = df['arthouse_score'].astype(int).value_counts().sort_index()
score_dist.index.name = 'arthouse_score'
score_dist.name = 'films'
display(score_dist.to_frame())

## 2. Coverage of the main signals

Six things we're likely to want for the next analysis.

In [ ]:
signals = {
    'IMDb rating':        df['imdbRating'].notna(),
    'Plot summary':       df['plotShort'].notna() | df['plotMedium'].notna() | df['plotLong'].notna(),
    'Keywords':           df['keywords'].notna(),
    'TMDB budget':        df['budget'].notna() & (df['budget'] != 0),
    'TMDB revenue':       df['revenue'].notna() & (df['revenue'] != 0),
    'MovieLens ratings':  df['ml_rating_count'].notna() & (df['ml_rating_count'] > 0),
}

coverage = pd.DataFrame({
    'films_with': [int(m.sum()) for m in signals.values()],
    'pct':        [round(m.mean() * 100, 1) for m in signals.values()],
}, index=list(signals.keys())).rename_axis('signal')
display(coverage)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(coverage.index, coverage['pct'], color='#3a5a78')
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel('% of arthouse films with this signal')
for i, (n, p) in enumerate(zip(coverage['films_with'], coverage['pct'])):
    ax.text(p + 1, i, f'{n:,}  ({p}%)', va='center', fontsize=9)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Joint coverage

How many arthouse films have the *combinations* a real analysis needs.

In [ ]:
combos = {
    'rating + budget':           signals['IMDb rating'] & signals['TMDB budget'],
    'rating + revenue':          signals['IMDb rating'] & signals['TMDB revenue'],
    'budget + revenue':          signals['TMDB budget'] & signals['TMDB revenue'],
    'rating + plot + keywords':  signals['IMDb rating'] & signals['Plot summary'] & signals['Keywords'],
}
joint = pd.DataFrame({
    'films': [int(m.sum()) for m in combos.values()],
    'pct':   [round(m.mean() * 100, 1) for m in combos.values()],
}, index=list(combos.keys())).rename_axis('combination')
display(joint)

## 4. Takeaway

Read off the numbers above to decide which angle is viable on this dataset.